## Dask and chunking

Lazy processing refers to the use of [out-of-core computation algorithms](https://en.wikipedia.org/wiki/External_memory_algorithm) to process very large data, which are usually too large to fit into the computer's memory at one time. The main idea is to chunk the data in pieces, small enough, that can be processed in memory.

HyperSpy internally uses the [dask library](https://docs.dask.org/en/latest/index.html), which implements the numpy interface to larger-than-memory or distributed environments. The typically workflow for processing data lazily which is on a disk:
1. "load" data from disk with a defined chunking
2. schedule operations
3. do the computation
 
Lets try this with a simple example:

In [ ]:
import dask.array as da

"Load" the data, by generating a big image with random data

In [ ]:
data = da.random.random(size=(50000, 50000), chunks=(1000, 1000))

Schedule operation, firsty taking the square root, then summing

In [ ]:
data_output = da.sqrt(data).sum()

**Steps 1 and 2 are very fast**, because nothing is actually done, other than initialising and scheduling the tasks to be performed.

Do the actual calculation, using `compute`

In [ ]:
data_output.compute()

### Chunking

An important aspect of lazy processing is **chunking**. This is how the data is organized inside files, like `cpu_sample_sped.zspy`.

In [ ]:
%matplotlib qt5
import hyperspy.api as hs

In [ ]:
s = hs.load("data/cpu_sample_sped.zspy", lazy=True)

For our 4-dimensional dataset here, the data is split into many smaller 4-dimensional chunks. To see this structure, we use `s.data`

In [ ]:
s.data

The important part is the chunk shape `(32, 32, 32, 32)`, which means each chunk consist of `32 x 32` probe positions, and `32 x 32` detector pixels. Each time we want to access something inside a chunk, we need to load the whole chunk into memory.

So for example, if we want to see what the value is for a single detector pixel at one specific probe position, we need to really get the full chunk. For example:

In [ ]:
s_single = s.inav[0, 0].isig[0, 0]
s_single.compute()

Requires just as much reading from the harddrive as reading the full chunk:

In [ ]:
s_single = s.inav[0:32, 0:32].isig[0:32, 0:32]
s_single.compute()

So when we're plotting, we're grabbing `32 x 32` probe positions, and `256 x 256` detector pixels (since we need to load the full detector image into memory to actually be able to see it).



When doing cropping and other types of processing, it is a good idea to keep the chunk sizes in mind. For example, grabbing exactly one chunk is much faster, than grabbing a chunk + 1.

In [ ]:
s.isig[128:128+32, 128:128+32].mean().compute()

In [ ]:
s.isig[128-1:128+33, 128-1:128+33].mean().compute()

Which is about 3 times slower! In this example here, both operations are pretty quick, but for larger datasets this can be a big timesaver!

But why split the detector into `32 x 32`? Giving 64 "detector" chunks? Typically we want to load the whole diffraction pattern into memory.

In [ ]:
s_rechunk = s.rechunk(nav_chunks=(32, 32), sig_chunks=(256, 256), inplace=False)

In [ ]:
s_rechunk

In [ ]:
s_rechunk_t = s_rechunk.T

In [ ]:
s_rechunk_t

Thus, this gives a chunking of `256 x 256` nav chunk (all the probe positions), and `32 x 32` signal chunks.

If we would try to plot this, the whole dataset would have to be loaded into memory.

Chunking is quite tricky, with there not being an "ideal" chunking strategy.